# BMW SORDI Smart Logistics Vision

Independent portfolio analysis of a public synthetic SORDI subset. This notebook is not a BMW tool and the numbers are not plant performance.

It reads the exported detection table and the dataset summary. KPI math comes from `src.analytics`, the same module the Streamlit app uses.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analytics import filter_detections, filter_false_negatives, load_table, summarize
summary_path = ROOT / "outputs" / "metrics" / "dataset_summary.json"
detections_path = ROOT / "outputs" / "tables" / "detections.parquet"
fn_path = ROOT / "outputs" / "tables" / "false_negatives.parquet"
if not summary_path.is_file() or not detections_path.is_file():
    raise FileNotFoundError("Run data preparation and evaluation before this notebook.")
dataset_summary = json.loads(summary_path.read_text(encoding="utf-8"))
detections = load_table(detections_path)
false_negatives = load_table(fn_path)
test_detections = filter_detections(detections, splits=["test"])
test_fn = filter_false_negatives(false_negatives, splits=["test"])
kpis = summarize(test_detections, test_fn)


## 2. Dataset summary

DR-1 only, four classes, at most 1,200 images, seed 42, 70/15/15 split.

In [2]:
print("images", dataset_summary["images"])
print("objects", dataset_summary["objects"])
print("images with class", dataset_summary["images_with_class"])
for split, info in dataset_summary["by_split"].items():
    print(split, info["images"], info["objects"])


images 1200
objects {'pallet': 5105, 'stillage': 9998, 'forklift': 689, 'dolly': 3868}
images with class {'pallet': 960, 'stillage': 1183, 'forklift': 662, 'dolly': 972}
train 840 {'pallet': 3686, 'stillage': 7071, 'forklift': 475, 'dolly': 2748}
val 180 {'pallet': 703, 'stillage': 1475, 'forklift': 106, 'dolly': 579}
test 180 {'pallet': 716, 'stillage': 1452, 'forklift': 108, 'dolly': 541}


## 3. Class distribution

In [3]:
pd.DataFrame(dataset_summary["by_split"]).T


,images,objects,images_with_class
train,840,"{'pallet': 3686, 'stillage': 7071, 'forklift':...","{'pallet': 677, 'stillage': 828, 'forklift': 4..."
val,180,"{'pallet': 703, 'stillage': 1475, 'forklift': ...","{'pallet': 140, 'stillage': 179, 'forklift': 1..."
test,180,"{'pallet': 716, 'stillage': 1452, 'forklift': ...","{'pallet': 143, 'stillage': 176, 'forklift': 1..."


## 4. Detection summary

Test-split predictions at the exported operating point.

In [4]:
{
    "total_detections": kpis["total_detections"],
    "mean_confidence": kpis["mean_confidence"],
    "fp_count": kpis["fp_count"],
    "fn_count": kpis["fn_count"],
    "detections_by_class": kpis["detections_by_class"],
    "class_mix": kpis["class_mix"],
}


{'total_detections': 2729,
 'mean_confidence': 0.676533625791168,
 'fp_count': 715,
 'fn_count': 794,
 'detections_by_class': {'pallet': 701,
  'stillage': 1398,
  'forklift': 112,
  'dolly': 518},
 'class_mix': {'pallet': 0.2568706485892268,
  'stillage': 0.5122755588127519,
  'forklift': 0.041040674239648224,
  'dolly': 0.18981311835837303}}

## 5. Precision / recall

These ratios use the one-to-one IoU matcher. They are not mAP.

In [5]:
pd.DataFrame({
    "precision": kpis["precision_by_class"],
    "recall": kpis["recall_by_class"],
    "fp": kpis["fp_by_class"],
    "fn": kpis["fn_by_class"],
})


,precision,recall,fp,fn
pallet,0.681883,0.670407,223,235
stillage,0.793991,0.766575,288,338
forklift,0.741071,0.768519,29,25
dolly,0.662162,0.636364,175,196


## 6. Confidence analysis

Low confidence means a score below 0.40.

In [6]:
print("low_confidence_rate", kpis["low_confidence_rate"])
print("average_confidence_by_class", kpis["average_confidence_by_class"])
test_detections["confidence"].describe()


low_confidence_rate 0.2121656284353243
average_confidence_by_class {'pallet': 0.5936464319130493, 'stillage': 0.7299666199646624, 'forklift': 0.7711181076509612, 'dolly': 0.6240454695859932}


count    2729.000000
mean        0.676534
std         0.246783
min         0.250060
25%         0.436638
50%         0.745867
75%         0.910952
max         0.990515
Name: confidence, dtype: float64

## 7. IoU analysis

In [7]:
pd.Series(kpis["iou_values"], name="iou").describe()


count    2014.000000
mean        0.869209
std         0.111563
min         0.500894
25%         0.811503
50%         0.908829
75%         0.954785
max         0.997767
Name: iou, dtype: float64

## 8. Error summary

In [8]:
error_path = ROOT / "outputs" / "metrics" / "error_analysis.json"
errors = json.loads(error_path.read_text(encoding="utf-8"))
pd.DataFrame(errors["examples"])


,file,image_id,reason,n_fp,n_fn,n_tp,min_match_iou,min_conf,confused
0,error_01_false_positive_1045.jpg,1045,false_positive,2,3,1,0.910499,0.264846,False
1,error_02_false_positive_1136.jpg,1136,false_positive,5,2,10,0.566937,0.271762,True
2,error_03_false_positive_1333.jpg,1333,false_positive,14,13,25,0.694543,0.251550,False
3,error_04_false_negative_1377.jpg,1377,false_negative,1,5,7,0.857594,0.488868,False
4,error_05_false_negative_1408.jpg,1408,false_negative,1,2,7,0.721933,0.284387,False
5,error_06_false_negative_1772.jpg,1772,false_negative,3,2,19,0.700477,0.353912,False
6,error_07_low_iou_398.jpg,398,low_iou,2,2,13,0.500894,0.250144,False
7,error_08_low_iou_2723.jpg,2723,low_iou,6,21,16,0.506609,0.294567,True
8,error_09_low_confidence_47.jpg,47,low_confidence,4,4,21,0.552806,0.250060,True
9,error_10_low_confidence_851.jpg,851,low_confidence,10,8,27,0.546371,0.250289,False


## 9. Limitations

The images are synthetic. The subset is capped at four classes and 1,200 frames, and the model is YOLO11n trained briefly on a 4 GB laptop GPU. Nothing in this notebook was measured in a BMW production environment, and the detector is not for safety-critical use.

## 10. Key observations

The next cell states only facts computed from the test export.

In [9]:
counts = kpis["detections_by_class"]
if counts:
    busiest = max(counts, key=counts.get)
    print(f"Most test detections: {busiest} ({counts[busiest]}).")
print(f"Test detections: {kpis['total_detections']}. False positives: {kpis['fp_count']}. False negatives: {kpis['fn_count']}.")
print(f"Low-confidence rate: {kpis['low_confidence_rate']}.")


Most test detections: stillage (1398).
Test detections: 2729. False positives: 715. False negatives: 794.
Low-confidence rate: 0.2121656284353243.
